# Week 5 In-Class Exercise: Naive Bayes Text Classification

In the demonstration, we saw **Naive Bayes** classify newsgroup posts: a fast, *generative* classifier built on Bayes' rule and a conditional-independence assumption, and the classic baseline for **text classification**.

In this exercise, you'll build the same kind of classifier yourself, then explore how the choices you make — the text representation, the NB variant, and the smoothing parameter — change the result.

> **Dataset: 20 Newsgroups.** Public Usenet posts. To keep things fast and focused, we'll use four benign topics: `comp.graphics`, `rec.autos`, `rec.sport.baseball`, and `sci.space`. Your classifier predicts which newsgroup a post came from.

You'll adapt the techniques from the demo, not invent new ones.

1. Load the text data
2. Turn documents into features with a **vectorizer**
3. Train a **Multinomial Naive Bayes** classifier and measure accuracy
4. Compare **CountVectorizer** vs **TF-IDF**
5. Analyze errors with a **confusion matrix**
6. Tune the **`alpha`** smoothing parameter
7. Classify **your own** sentences

### Useful API References

| Task | Documentation |
|------|---------------|
| Load text data | [fetch_20newsgroups](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html) |
| Word counts | [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) |
| TF-IDF | [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) |
| Multinomial NB | [MultinomialNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html) |
| Complement NB | [ComplementNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.ComplementNB.html) |
| Pipelines | [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) |
| Accuracy / report | [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html), [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) |
| Confusion matrix | [ConfusionMatrixDisplay](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html) |

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week5/week5_naive_bayes_inclass_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=11)

## Step 1: Load the Data (Provided)

We fetch the train and test splits and strip headers/footers/quotes so the model learns from the message content, not metadata. This cell is provided — just run it. (The first run downloads ~14 MB.)

In [ ]:
from sklearn.datasets import fetch_20newsgroups

categories = ['comp.graphics', 'rec.autos', 'rec.sport.baseball', 'sci.space']
remove = ('headers', 'footers', 'quotes')

train = fetch_20newsgroups(subset='train', categories=categories,
                           remove=remove, random_state=42)
test = fetch_20newsgroups(subset='test', categories=categories,
                          remove=remove, random_state=42)

print('Training documents:', len(train.data))
print('Test documents:    ', len(test.data))
print('Classes:', train.target_names)
print('\n--- Example document ---')
print('LABEL:', train.target_names[train.target[0]])
print(train.data[0][:300])

## Step 2: Turn Text into Features

A classifier needs numbers, not strings. `CountVectorizer` builds a "bag of words": each document becomes a vector of word counts.

**Your turn!** Fit a `CountVectorizer(stop_words='english')` on the training text and report the shape of the resulting document-term matrix and the vocabulary size.

**Hint (from the demo):**
```python
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(stop_words='english')
X_train_counts = vectorizer.fit_transform(train.data)
X_train_counts.shape
```

In [ ]:
# TODO: Fit a CountVectorizer on train.data. Print the document-term matrix
#       shape and the vocabulary size.
# Your code here:




**Question:** The document-term matrix has thousands of columns (one per vocabulary word) but each individual post only uses a few dozen words. What does that tell you about the matrix — is it mostly full or mostly empty? Why is that a good match for Naive Bayes rather than a problem?

*Your answer:*



## Step 3: Train Multinomial Naive Bayes

`MultinomialNB` is the NB variant built for count data. Bundle the vectorizer and classifier in a `Pipeline` so the same transformation applies to train and test.

**Your turn!** Build and fit a `make_pipeline(CountVectorizer(stop_words='english'), MultinomialNB())`, then print the **test accuracy**.

**Hint (from the demo):**
```python
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
nb_clf = make_pipeline(CountVectorizer(stop_words='english'), MultinomialNB())
nb_clf.fit(train.data, train.target)
```

In [ ]:
# TODO: Build the pipeline, fit on the training data, and print test accuracy.
# Your code here:




**Question:** This is a 4-class problem. What accuracy would a classifier get by always guessing the same single class (roughly)? How does your Naive Bayes accuracy compare?

*Your answer:*



## Step 4: CountVectorizer vs TF-IDF

Raw counts over-reward common words. **TF-IDF** re-weights each word by how *informative* it is (rare, topic-specific words count more).

**Your turn!** Build a second pipeline using `TfidfVectorizer(stop_words='english')` instead of `CountVectorizer`, and print its test accuracy next to the CountVectorizer one from Step 3.

**Hint (from the demo):**
```python
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_clf = make_pipeline(TfidfVectorizer(stop_words='english'), MultinomialNB())
```

In [ ]:
# TODO: Build a TF-IDF + MultinomialNB pipeline; print its test accuracy
#       alongside the CountVectorizer result.
# Your code here:




**Question:** Did TF-IDF help, hurt, or make little difference here? Give one reason TF-IDF often improves a text classifier compared to raw counts.

*Your answer:*



## Step 5: Error Analysis

**Your turn!** Using your TF-IDF classifier's predictions on the test set, display a **confusion matrix** (with `display_labels=train.target_names`) and print a `classification_report`.

**Hint (from the demo):**
```python
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
ConfusionMatrixDisplay.from_predictions(test.target, y_pred,
                                        display_labels=train.target_names,
                                        xticks_rotation=45)
```

In [ ]:
# TODO: Display the confusion matrix and print the classification report
#       for your TF-IDF classifier.
# Your code here:




**Question:** Which two newsgroups get confused most often? Suggest a reason based on the *vocabulary* those topics might share.

*Your answer:*



## Step 6: Tune the Smoothing Parameter `alpha`

Naive Bayes multiplies per-word probabilities; a word unseen in a class would give probability 0 and wipe out the whole product. **Additive smoothing** adds `alpha` to every count to prevent that. `alpha=1.0` is the default.

**Your turn!** Loop over `alpha` in `[0.001, 0.01, 0.1, 1.0, 10.0]` for a TF-IDF + `MultinomialNB(alpha=...)` pipeline and print the test accuracy for each. Which `alpha` is best?

**Hint (from the demo):**
```python
for alpha in [0.001, 0.01, 0.1, 1.0, 10.0]:
    model = make_pipeline(TfidfVectorizer(stop_words='english'),
                          MultinomialNB(alpha=alpha))
```

In [ ]:
# TODO: Sweep alpha and print test accuracy for each value.
# Your code here:




**Question:** What happens to accuracy when `alpha` is very small (like 0.001) versus very large (like 10)? Explain in terms of trusting rare words too much versus washing out real signal.

*Your answer:*



## Step 7: Classify Your Own Sentences

**Your turn!** Write **four** short sentences of your own — one clearly about each topic (computer graphics, cars, baseball, space) — put them in a list, and have your best classifier predict the topic of each. Print each sentence next to its predicted label.

**Hint:** The pipeline predicts directly from raw strings: `best_clf.predict(my_sentences)` returns label indices; map them through `train.target_names`.

In [ ]:
# TODO: Create a list of 4 sentences (one per topic) and print each with its
#       predicted newsgroup label.
# Your code here:




**Question:** Did the classifier get all four of your sentences right? If any were wrong (or you could imagine one being wrong), what about the sentence's wording might have thrown it off?

*Your answer:*



## Reflection Questions

Answer each question in the cell below it (1-3 sentences each).

**Q1:** Naive Bayes assumes words are *independent given the class*. Give an example of two words whose appearances are clearly **not** independent in real text. Why does the classifier still work well despite this false assumption?

*Your answer:*



**Q2:** This week's other topic was SVMs. Naive Bayes trains almost instantly and needs little data; an SVM with an RBF kernel is slower. Name one situation where you'd reach for Naive Bayes first, and one where an SVM might be worth the extra cost.

*Your answer:*



**Q3:** We used `MultinomialNB` for word counts. Why would `GaussianNB` be the wrong choice for this bag-of-words data, and what kind of data *is* `GaussianNB` designed for?

*Your answer:*



**Q4:** We stripped the headers, footers, and quoted text from the posts. How do you think accuracy would change if we left them in, and would that be a fair measure of the classifier's real ability to understand the message content?

*Your answer:*

